# 04. Model B Training — Income & Affordability Score

This notebook trains a Gradient Boosting Regressor to estimate the income adequacy score (Model B).

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split
import pickle

# Load data
df = pd.read_csv('../backend/data/synthetic_loan_data.csv')

# Target: income_adequacy_label (top 20% = 1.0, etc.)
df['income_adequacy_label'] = pd.qcut(df['applicant_income'], 5, labels=[0.2, 0.4, 0.6, 0.8, 1.0]).astype(float)

# Features for Model B
def compute_b_features(row):
    govt_map = {"BPL": 1.0, "APL": 0.5, "None": 0.0}
    stab_map = {"Salaried": 3.0, "Self-employed": 2.0, "Farmer": 1.5, "Daily wage": 1.0}
    dti = min(20.0, row['loan_amount'] / row['applicant_income'] if row['applicant_income'] > 0 else 20.0)
    total_inc = row['applicant_income'] + row['coapplicant_income']
    burden = min(5.0, row['dependents'] / total_inc if total_inc > 0 else (5.0 if row['dependents'] > 0 else 0))
    return pd.Series({
        "electricity_bill_avg": row['electricity_bill_avg'],
        "mobile_recharge_avg": row['mobile_recharge_avg'],
        "mobile_recharge_frequency": row['mobile_recharge_frequency'],
        "govt_category_encoded": govt_map.get(row['govt_socioeconomic_category'], 0.0),
        "applicant_income_log": np.log1p(row['applicant_income']),
        "coapplicant_income_log": np.log1p(row['coapplicant_income']),
        "debt_to_income": dti,
        "household_burden": burden,
        "employment_stability": stab_map.get(row['employment_type'], 1.0)
    })

X = df.apply(compute_b_features, axis=1).fillna(0.5)
y = df['income_adequacy_label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = GradientBoostingRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

with open('../models/model_b_income.pkl', 'wb') as f:
    pickle.dump(model, f)
print("Model B saved.")